NOI SIAMO IL GRUPPO B CAZZOOOOO SIIIII

Vari import fatti durante le serie

In [1]:
import pandas as pd
import numpy as np
import os
from collections import Counter
import re
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

Read CSV

In [2]:
df = pd.read_csv("data/Video Games Sales (1980-2024) - Raw.csv")
df_cleaned = pd.read_csv("data/Video_Games_Sales_Cleaned.csv")

Righe e colonne del CSV, e informazioni sulle colonne

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 64016 entries, 0 to 64015
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   img           64016 non-null  str    
 1   title         64016 non-null  str    
 2   console       64016 non-null  str    
 3   genre         64016 non-null  str    
 4   publisher     64016 non-null  str    
 5   developer     63999 non-null  str    
 6   critic_score  6678 non-null   float64
 7   total_sales   18922 non-null  float64
 8   na_sales      12637 non-null  float64
 9   jp_sales      6726 non-null   float64
 10  pal_sales     12824 non-null  float64
 11  other_sales   15128 non-null  float64
 12  release_date  56965 non-null  str    
 13  last_update   17879 non-null  str    
dtypes: float64(6), str(8)
memory usage: 6.8 MB


Numero totale di giochi

In [33]:
# Numero giochi
len(df["title"].unique())

39798

In [15]:
# Istogramma vendite globali
fig = px.histogram(
    df,
    x="total_sales",
    log_y=True,
    nbins=15,
    title="Distribution of Global Sales",
    labels={
        "total_sales": "Total sales [millions of copies]",
        "count": "Number of games"
    }
)
fig.update_traces(xbins_start=0)
fig.show()

In [47]:
# df.loc[df["total_sales"].idxmax()]
df.sort_values(by="total_sales", ascending=False)

,img,title,console,genre,publisher,developer,critic_score,total_sales,na_sales,jp_sales,pal_sales,other_sales,release_date,last_update
0,/games/boxart/full_6510540AmericaFrontccc.jpg,Grand Theft Auto V,PS3,Action,Rockstar Games,Rockstar North,9.4,20.32,6.37,0.99,9.85,3.12,17-09-2013,NaN
1,/games/boxart/full_5563178AmericaFrontccc.jpg,Grand Theft Auto V,PS4,Action,Rockstar Games,Rockstar North,9.7,19.39,6.06,0.60,9.71,3.02,18-11-2014,03-01-2018
2,/games/boxart/827563ccc.jpg,Grand Theft Auto: Vice City,PS2,Action,Rockstar Games,Rockstar North,9.6,16.15,8.41,0.47,5.49,1.78,28-10-2002,NaN
3,/games/boxart/full_9218923AmericaFrontccc.jpg,Grand Theft Auto V,X360,Action,Rockstar Games,Rockstar North,NaN,15.86,9.06,0.06,5.33,1.42,17-09-2013,NaN
4,/games/boxart/full_4990510AmericaFrontccc.jpg,Call of Duty: Black Ops 3,PS4,Shooter,Activision,Treyarch,8.1,15.09,6.18,0.41,6.05,2.44,06-11-2015,14-01-2018
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64011,/games/boxart/full_2779838AmericaFrontccc.jpg,XBlaze Lost: Memories,PC,Visual Novel,Aksys Games,Arc System Works,NaN,NaN,NaN,NaN,NaN,NaN,11-08-2016,28-01-2019
64012,/games/boxart/full_8031506AmericaFrontccc.jpg,"Yoru, Tomosu",PS4,Visual Novel,Nippon Ichi Software,Nippon Ichi Software,NaN,NaN,NaN,NaN,NaN,NaN,30-07-2020,09-05-2020
64013,/games/boxart/full_6553045AmericaFrontccc.jpg,"Yoru, Tomosu",NS,Visual Novel,Nippon Ichi Software,Nippon Ichi Software,NaN,NaN,NaN,NaN,NaN,NaN,30-07-2020,09-05-2020
64014,/games/boxart/full_6012940JapanFrontccc.png,Yunohana SpRING! ~Mellow Times~,NS,Visual Novel,Idea Factory,Otomate,NaN,NaN,NaN,NaN,NaN,NaN,28-02-2019,24-02-2019


In [46]:
# Giochi aggregati così da evitare ripetiizioni per via di distribuzioni su console diverse
sales_by_title = (
    df.groupby("title", as_index=False)[
        ["total_sales", "na_sales", "jp_sales", "pal_sales", "other_sales"]
    ]
    .sum()
)
sales_by_title.sort_values(by="total_sales", ascending=False)

,title,total_sales,na_sales,jp_sales,pal_sales,other_sales
13724,Grand Theft Auto V,64.29,26.19,1.66,28.14,8.32
5266,Call of Duty: Black Ops,30.99,17.65,0.59,9.45,3.31
5281,Call of Duty: Modern Warfare 3,30.71,15.57,0.62,11.26,3.26
5273,Call of Duty: Black Ops II,29.59,14.12,0.72,11.08,3.67
5277,Call of Duty: Ghosts,28.80,15.06,0.49,9.60,3.65
...,...,...,...,...,...,...
39793,yOm,0.00,0.00,0.00,0.00,0.00
39794,yOm_fury,0.00,0.00,0.00,0.00,0.00
8361,Derby Time,0.00,0.00,0.00,0.00,0.00
39796,じんるいのみなさまへ,0.00,0.00,0.00,0.00,0.00


In [ ]:
sales_cols = ["total_sales", "na_sales", "jp_sales", "pal_sales", "other_sales"]

df_sales = df.copy()
df_sales[sales_cols] = df_sales[sales_cols].fillna(0)

complete_sales_by_title = (
    df_sales.groupby("title", as_index=False)
    .agg({
        "genre": "first",
        "publisher": "first",
        "developer": "first",
        "critic_score": "mean", # media su score 
        "release_date": "min", # prima uscita
        "total_sales": "sum",
        "na_sales": "sum",
        "jp_sales": "sum",
        "pal_sales": "sum",
        "other_sales": "sum"
    })
    .rename(columns={"console": "n_consoles"})
)
complete_sales_by_title.sort_values(by="total_sales", ascending=False)

,title,genre,publisher,developer,critic_score,release_date,total_sales,na_sales,jp_sales,pal_sales,other_sales
13724,Grand Theft Auto V,Action,Rockstar Games,Rockstar North,9.366667,01-12-2021,64.29,26.19,1.66,28.14,8.32
5266,Call of Duty: Black Ops,Shooter,Activision,Treyarch,8.287500,03-05-2011,30.99,17.65,0.59,9.45,3.31
5281,Call of Duty: Modern Warfare 3,Shooter,Activision,Infinity Ward,7.500000,08-11-2011,30.71,15.57,0.62,11.26,3.26
5273,Call of Duty: Black Ops II,Shooter,Activision,Treyarch,8.375000,13-11-2012,29.59,14.12,0.72,11.08,3.67
5277,Call of Duty: Ghosts,Shooter,Activision,Infinity Ward,7.833333,05-11-2013,28.80,15.06,0.49,9.60,3.65
...,...,...,...,...,...,...,...,...,...,...,...
39793,yOm,Action,Microsoft,jojito,NaN,21-10-2009,0.00,0.00,0.00,0.00,0.00
39794,yOm_fury,Action,Microsoft,jojito,NaN,12-11-2009,0.00,0.00,0.00,0.00,0.00
8361,Derby Time,Sports,Sony Computer Entertainment,Sony Computer Entertainment,NaN,21-04-2005,0.00,0.00,0.00,0.00,0.00
39796,じんるいのみなさまへ,Visual Novel,Unknown,Nippon Ichi Software,NaN,NaN,0.00,0.00,0.00,0.00,0.00


In [56]:
# Istogramma vendite globali
fig = px.histogram(
    complete_sales_by_title,
    x="total_sales",
    log_y=True,
    nbins=15,
    title="Distribution of Global Sales",
    labels={
        "total_sales": "Total sales [millions of copies]",
        "count": "Number of games"
    }
)
fig.update_traces(xbins_start=0)
fig.show()

In [14]:
# Top 10 generi
top_genres = df["genre"].value_counts().head(10).reset_index()

fig = px.bar(
    top_genres,
    x="genre",
    y="count",
    title="Top 10 Genres by Number of Games",
    labels={
        "genre": "Genre",
        "count": "Number of games"
    }
)

fig.show()

In [34]:
# Ranking frequenza generi
genres = df["genre"].value_counts().reset_index()

#genres = genres.sort_values(by="count", ascending=True)

fig = px.bar(
    genres,
    x="count",
    y="genre",
    title="Ranking Genres by Number of Games",
    labels={
        "genre": "Genre",
        "count": "Number of games"
    }
)

fig.update_layout(
    yaxis = {'categoryorder': 'total ascending'},
    height = 600
)

fig.show()

In [57]:
df["console"].unique()

<StringArray>
[   'PS3',    'PS4',    'PS2',   'X360',   'XOne',     'PC',    'PSP',
    'Wii',     'PS',     'DS',   '2600',    'GBA',    'NES',     'XB',
    'PSN',    'GEN',    'PSV',     'DC',    'N64',    'SAT',   'SNES',
    'GBC',     'GC',     'NS',    '3DS',     'GB',   'WiiU',     'WS',
     'VC',     'NG',     'WW',    'SCD',    'PCE',    'XBL',    '3DO',
     'GG',    'OSX',    'Mob',   'PCFX', 'Series',    'All',    'iOS',
   '5200',    'And',   'DSiW',   'Lynx',  'Linux',     'MS',    'ZXS',
   'ACPC',   'Amig',   '7800',    'DSi',     'AJ',   'WinP',   'iQue',
    'GIZ',     'VB',   'Ouya',  'NGage',    'AST',    'MSD',   'S32X',
     'XS',    'PS5',    'Int',     'CV',    'Arc',    'C64',    'FDS',
    'MSX',     'OR',   'C128',    'CDi',   'CD32',    'BRW',    'FMT',
   'ApII',    'Aco',   'BBCM',   'TG16']
Length: 81, dtype: str

In [71]:
df_clean = df[["critic_score", "total_sales"]].dropna()
corr = df_clean["critic_score"].corr(df_clean["total_sales"])

fig = px.density_heatmap(
    df_clean,
    x="critic_score",
    y="total_sales",
    nbinsx=20,
    nbinsy=20,
    title="Density Heatmap: Critic Score vs Total Sales",
    labels={
        "critic_score": "Critic Score",
        "total_sales": "Total Sales (millions)"
    }
)

fig.update_layout(
    height = 600
)

fig.show()

In [ ]:
corr = df_clean["critic_score"].corr(df_clean["total_sales"])

fig = px.scatter(
    df_clean,
    x="critic_score",
    y="total_sales",
    log_y=True,
    trendline="ols",
    title=f"Critic Score vs Total Sales (corr = {corr:.2f})",
    labels={
        "critic_score": "Critic Score",
        "total_sales": "Total Sales (millions)"
    },
    opacity=0.6
)

fig.show()